<a href="https://colab.research.google.com/github/Doop318/trading-ecosystem-project/blob/main/algorithmic_trading_backtester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class FinancialAsset:
    """Manages raw asset historical price data (Encapsulation)."""
    def __init__(self, ticker, csv_path):
        self.ticker = ticker
        # Read historical CSV data generated from the yfinance step
        self.data = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        print(f"📦 Asset [{self.ticker}] loaded with {len(self.data)} daily price points.")

class TradingStrategy:
    """Base abstract interface class for all trading systems (Abstraction / Inheritance)."""
    def __init__(self, name):
        self.name = name

    def generate_signals(self, asset: FinancialAsset) -> pd.DataFrame:
        """Blueprint method to override in child strategy classes."""
        raise NotImplementedError("Each strategy must override generate_signals()")

class MovingAverageCrossover(TradingStrategy):
    """A moving average crossover strategy that inherits from TradingStrategy (Polymorphism)."""
    def __init__(self, fast_period=20, slow_period=50):
        super().__init__(name=f"MA Crossover ({fast_period}/{slow_period})")
        self.fast_period = fast_period
        self.slow_period = slow_period

    def generate_signals(self, asset: FinancialAsset) -> pd.DataFrame:
        """Generates clear buy (1) and sell (-1) trading triggers based on moving averages."""
        df = asset.data.copy()
        signals = pd.DataFrame(index=df.index)
        signals['Close'] = df['Close']

        # Calculate moving averages
        signals['Fast_MA'] = df['Close'].rolling(window=self.fast_period).mean()
        signals['Slow_MA'] = df['Close'].rolling(window=self.slow_period).mean()

        # Core Strategy Logic: 1 when Fast MA > Slow MA, else 0
        signals['Position_Signal'] = 0.0
        signals['Position_Signal'][self.fast_period:] = np.where(
            signals['Fast_MA'][self.fast_period:] > signals['Slow_MA'][self.fast_period:], 1.0, 0.0
        )

        # Extract direct trade executions (-1 = Sell Trigger, 1 = Buy Trigger)
        signals['Trade_Action'] = signals['Position_Signal'].diff()
        return signals

# ==========================================================
# RUNNING THE SIMULATION ENVIRONMENT
# ==========================================================
try:
    # 1. Instantiate the asset object (Make sure your file path matches your saved CSV name)
    apple_stock = FinancialAsset(ticker="AAPL", csv_path="data/AAPL_historical.csv")

    # 2. Instantiate the technical OOP strategy engine
    ma_strategy = MovingAverageCrossover(fast_period=20, slow_period=50)

    # 3. Generate strategic actions
    trade_signals = ma_strategy.generate_signals(apple_stock)

    # 4. View results inside your notebook layout
    print("\n🔍 Recent Trading Signals Log:")
    print(trade_signals[['Close', 'Fast_MA', 'Slow_MA', 'Trade_Action']].tail(15))

    # 5. Visual Validation Chart
    plt.figure(figsize=(14, 7))
    plt.plot(trade_signals['Close'], label='AAPL Close Price', color='black', alpha=0.5)
    plt.plot(trade_signals['Fast_MA'], label='20-Day Fast MA', color='blue', linestyle='--')
    plt.plot(trade_signals['Slow_MA'], label='50-Day Slow MA', color='orange', linestyle='--')

    # Draw Buy markers
    plt.scatter(trade_signals.loc[trade_signals.Trade_Action == 1.0].index,
                trade_signals.Fast_MA[trade_signals.Trade_Action == 1.0],
                marker='^', color='green', s=100, label='BUY SIGNAL')

    # Draw Sell markers
    plt.scatter(trade_signals.loc[trade_signals.Trade_Action == -1.0].index,
                trade_signals.Fast_MA[trade_signals.Trade_Action == -1.0],
                marker='v', color='red', s=100, label='SELL SIGNAL')

    plt.title(f"Quantitative OOP Trading Signals Execution - {ma_strategy.name}")
    plt.xlabel('Timeline')
    plt.ylabel('Asset Value ($)')
    plt.legend(loc='best')
    plt.grid(True)
    plt.show()

except Exception as e:
    print(f"⚠️ Simulation setup required confirmation: {e}")
    print("Ensure you ran your first data collection cell successfully to create 'data/AAPL_historical.csv' file.")


⚠️ Simulation setup required confirmation: [Errno 2] No such file or directory: 'data/AAPL_historical.csv'
Ensure you ran your first data collection cell successfully to create 'data/AAPL_historical.csv' file.
